# 06 — Group Relative Policy Optimization (GRPO)

**Módulo:** `src/llm_rlhf/grpo.py` → `GRPOTrainer`

GRPO (Shao et al., 2024) es el algoritmo detrás de DeepSeek-R1. Conserva el *clipped surrogate* de PPO pero **descarta la función de valor**. La idea es pequeña y la implementación es corta.

## Setup (Colab o entorno nuevo)

Esta primera celda es **idempotente**: si `llm_rlhf` ya está instalado, no hace nada. En Colab, la primera vez clona el repositorio y lo instala en modo editable.

El repositorio apuntado es [emiliomunozai/LLM_RLHF](https://github.com/emiliomunozai/LLM_RLHF).

In [ ]:
# --- Colab / fresh-environment setup ---------------------------------
# No-op if llm_rlhf is already importable (e.g. local uv environment).
import os, sys, subprocess

REPO_URL = "https://github.com/emiliomunozai/LLM_RLHF.git"
REPO_DIR = "LLM_RLHF"

try:
    import llm_rlhf  # noqa: F401
except ImportError:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    print("Installed llm_rlhf. If torch/transformers were upgraded, restart the runtime now.")

## ¿Qué tiene de malo la función de valor de PPO?

En RL clásico, el crítico $V_\phi(s)$ es directo — los entornos emiten recompensas por paso y la cabeza de valor aprende la suma descontada.

En modelado de lenguaje es mucho más difícil:

- Las recompensas son **dispersas** (un escalar al final de una respuesta larga).
- Las recompensas son **ruidosas** (el modelo de recompensa es en sí imperfecto).
- Las secuencias son **largas** (cientos de tokens), así que la asignación de crédito es lenta.

La cabeza de valor termina con alta varianza, y un baseline de alta varianza da ventajas de alta varianza, lo que desestabiliza la actualización del policy. GRPO esquiva el problema *no aprendiendo* función de valor en absoluto.

## Ventajas relativas al grupo

Para cada prompt $x$, muestrea $G$ respuestas (típicamente 4–8) del policy. Puntúalas todas con el modelo de recompensa para obtener $r_1, \dots, r_G$. Entonces:

$$A_i = \frac{r_i - \mu_G}{\sigma_G}, \qquad \mu_G = \frac{1}{G}\sum_j r_j, \; \sigma_G = \text{std}_j(r_j)$$

El baseline es la **media del grupo**, no una función de valor aprendida. Es esencialmente la misma idea que REINFORCE-con-baseline (`A = R − b`), pero con el baseline calculado *por-prompt* a partir de muestras on-policy. La varianza la usamos para *blanquear* la señal — el mismo z-score que ya viste en PPO.

### El mismo escalar para todos los tokens de la respuesta

Una vez calculado $A_i$ para la respuesta $i$, ese **mismo escalar se difunde por cada token** de esa respuesta. Ya no hay ventaja por token como en GAE.

¿Por qué tiene sentido? Porque la recompensa del RM se entrega *al final* de la respuesta y la asignación de crédito por token sería, en el mejor caso, una conjetura del crítico. GRPO admite la honestidad del problema: no sabemos qué token específico hizo a la respuesta buena o mala, así que castigamos o premiamos al *trayecto entero* por igual. Es la asignación de crédito implícita en REINFORCE: si el resultado fue mejor que el promedio del grupo, sube la log-prob de toda la trayectoria.

Esto significa que el clipping de PPO sigue actuando *por token* (el ratio $\rho_t$ se computa token a token), pero la **señal** $A_i$ es constante a lo largo de la respuesta.

In [ ]:
import torch

def group_advantage(rewards, eps=1e-6):
    return (rewards - rewards.mean()) / (rewards.std() + eps)

r = torch.tensor([1.4, -0.3, 0.8, -0.7])  # 4 sampled responses, varying quality
print('rewards    :', r.tolist())
print('advantages :', group_advantage(r).tolist())

### Visualizando ventajas relativas al grupo

La z-score es la operación clave. Veamos qué hace con tres tipos de grupo: uno con alta varianza, uno donde todas las respuestas están agrupadas, y uno con un único ganador claro.

Mira las dos filas: la de arriba son las recompensas crudas, la de abajo son las ventajas. La línea roja marca la media del grupo (el baseline).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Show three different groups and what z-scoring does to them.
groups = {
    'high variance':   torch.tensor([+2.5, -1.0, +0.8, -2.2, +1.5, -1.8, +0.3, -0.1]),
    'tight cluster':   torch.tensor([+0.5, +0.4, +0.6, +0.3, +0.7, +0.4, +0.5, +0.6]),
    'one clear winner': torch.tensor([-0.2, -0.3, -0.1, +2.0, -0.4, -0.5, -0.2, -0.3]),
}

fig, axes = plt.subplots(2, 3, figsize=(13, 5), sharex=True)
for col, (name, rewards) in enumerate(groups.items()):
    adv = group_advantage(rewards).numpy()
    axes[0, col].bar(range(len(rewards)), rewards.numpy(), color='#4a90d9')
    axes[0, col].set_title(f'rewards — {name}')
    axes[0, col].axhline(rewards.mean().item(), color='red', linestyle='--', alpha=0.7, label=f'μ = {rewards.mean():.2f}')
    axes[0, col].legend(fontsize=8)
    axes[1, col].bar(range(len(rewards)), adv, color='#ff7f50')
    axes[1, col].set_title('advantage = (r − μ) / σ')
    axes[1, col].axhline(0, color='black', linewidth=0.5)
for ax in axes.flat:
    ax.set_xlabel('response in group')
plt.tight_layout()
plt.show()

## El objetivo

El mismo clipping de PPO, $A$ relativo al grupo, y una penalización KL explícita contra la referencia congelada:

$$L_{\text{GRPO}} = -\mathbb{E}\!\left[ \min\big(\rho_t A_i, \, \text{clip}(\rho_t, 1-\epsilon, 1+\epsilon) A_i \big) \right] + \beta \, \mathbb{E}\!\left[ \widehat{\text{KL}}_t \right]$$

Nota que la penalización KL ahora va **fuera** de la recompensa (en PPO estaba doblada dentro de la señal de recompensa por token). Funcionalmente similar, pero más limpio de leer.

## Qué desaparece, qué se queda

| Componente | PPO | GRPO |
|---|---|---|
| Modelo de recompensa | sí | sí |
| Referencia congelada | sí | sí |
| Rollouts | 1 por prompt | $G$ por prompt |
| Función de valor | sí | no |
| GAE | sí | no (la ventaja es un escalar por respuesta) |
| Clipped surrogate | sí | sí |
| Penalización KL | por token, dentro de la recompensa | por token, término separado |

La pérdida en `grpo.py` es más corta que `ppo/trainer.py` *y* más fácil de depurar — la ventaja es ahora un único tensor de longitud $G$ en vez de una matriz `[B, T]`.

## Coste frente a PPO

GRPO cuesta más *cómputo por prompt* (muestreas $G$ respuestas en lugar de 1). A cambio, te ahorras entrenar la función de valor y evitas GAE por completo. En tareas de razonamiento donde los rollouts son caros pero los grupos son manejables, GRPO suele dominar empíricamente a PPO.

Un modelo mental razonable:

- PPO usa un *baseline paramétrico* (la cabeza de valor) — requiere entrenamiento.
- GRPO usa un *baseline no paramétrico* (la media del grupo) — gratis, pero requiere $G$ muestras.

### El compromiso de `group_size`

La estabilidad del baseline depende de $G$. Si $G$ es demasiado pequeño, la media y la desviación estándar del grupo son ruidosas y las ventajas no informan bien. Si $G$ es demasiado grande, gastas cómputo sin ganar mucho.

La siguiente celda hace un experimento sintético: muestrea repetidamente $G$ recompensas de una $\mathcal{N}(0, 1)$ y mira cómo varía la ventaja del *mejor* del grupo. Observa cómo cae la incertidumbre con $G$ — y cómo el coste crece linealmente en $G$.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# How stable is the group-relative baseline as we change G?
# Repeatedly sample G rewards from the *same* underlying distribution and
# look at the standard deviation of the *advantage of the best response*.
rng = np.random.default_rng(0)
group_sizes = [2, 4, 8, 16, 32]
n_trials = 2000

results = []
for G in group_sizes:
    top_advs = []
    for _ in range(n_trials):
        r = torch.tensor(rng.normal(0, 1, size=G).astype('float32'))
        a = group_advantage(r)
        top_advs.append(a.max().item())
    results.append((G, np.mean(top_advs), np.std(top_advs)))

xs   = [g for g, _, _ in results]
mean = [m for _, m, _ in results]
std  = [s for _, _, s in results]

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(xs, mean, yerr=std, marker='o', capsize=4, linewidth=2)
ax.set_xscale('log', base=2)
ax.set_xticks(xs)
ax.set_xticklabels(xs)
ax.set_xlabel('group size G')
ax.set_ylabel('top-of-group advantage (mean ± std)')
ax.set_title('Variance of the advantage estimate shrinks with G\n(but compute grows linearly in G)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Punto de partida: SFT + Reward Model

Como PPO, GRPO encadena dos artefactos previos:

- **Policy inicial** = adaptador LoRA de `02_sft` (`checkpoints/sft/adapter`).
- **Modelo de recompensa** = checkpoint de `03_reward_model` (`checkpoints/reward/`). Sigue congelado.

La diferencia operativa con PPO está en el *rollout*: para cada prompt, GRPO muestrea **G respuestas** (no una). Esas G respuestas son simultáneamente los datos sobre los que se calcula la ventaja y los datos sobre los que se actualiza el policy. Si subes la `temperatura` del muestreo dentro del grupo, induces más diversidad y le das más señal al baseline relativo — pero también añades varianza.

La siguiente celda muestra el punto de partida del policy con la misma carga idempotente.

In [ ]:
import os
from llm_rlhf import PretrainedLLM, CANONICAL_PROMPT, EVAL_PROMPTS
from llm_rlhf.eval import SFT_ADAPTER_PATH, REWARD_CHECKPOINT_PATH

llm = PretrainedLLM()

sft_path = os.path.join("..", SFT_ADAPTER_PATH)
rm_path  = os.path.join("..", REWARD_CHECKPOINT_PATH)
print(f"SFT adapter found: {os.path.exists(sft_path)}  ({sft_path})")
print(f"Reward checkpoint found: {os.path.exists(rm_path)}  ({rm_path})")
if os.path.exists(sft_path):
    llm.load_adapter(sft_path)

print("\n--- Starting point for GRPO (SFT model if loaded, else base) ---")
for p in EVAL_PROMPTS:
    print(f"PROMPT:     {p}")
    print(f"COMPLETION: {llm.generate(p)}\n")

# A small peek at what one GRPO 'group' looks like for the canonical prompt:
# four samples from the same starting policy. These are the four responses
# that, in real training, would receive scores from the RM and then be
# z-scored against each other to produce per-response advantages.
print(f"--- Example group (G=4) for: {CANONICAL_PROMPT!r} ---")
for i in range(4):
    print(f"[{i}] {llm.generate(CANONICAL_PROMPT)[:150]}")

In [ ]:
from llm_rlhf.grpo import GRPOTrainer, GRPOConfig
from llm_rlhf.config import load_toml

cfg = GRPOConfig(**load_toml('../configs/grpo.toml'))
print(cfg)

## Ejercicio: colapso y dominancia

Hay dos casos extremos que vale la pena reconocer:

1. **Todas las respuestas son aproximadamente iguales** — la varianza del grupo se acerca a cero, las ventajas explotan numéricamente (el `eps` las protege) o se vuelven cero. En cualquier caso, GRPO no aprende nada de este grupo.
2. **Una respuesta es dramáticamente mejor que el resto** — el ganador domina la actualización; los demás contribuyen ruido.

La siguiente celda construye varios de estos casos. Después, piensa:

- ¿Qué configuración maximiza la *magnitud* del update?
- ¿Qué configuración es *invisible* para GRPO (gradiente cero)?
- ¿Cómo te conectaría esto al uso de *temperatura* alta en el muestreo para forzar diversidad dentro del grupo?

In [ ]:
# Exercise: collapse and dominance
# What happens to the advantages when one response is dramatically better than the rest?
# What about when all responses are roughly equal? Both are signals worth recognising.
import torch

cases = {
    'all equal':          torch.tensor([0.5, 0.5, 0.5, 0.5]),
    'one big outlier':    torch.tensor([0.0, 0.0, 0.0, 5.0]),
    'pair of winners':    torch.tensor([0.0, 2.0, 0.0, 2.0]),
    'pair of losers':     torch.tensor([0.0, -2.0, 0.0, -2.0]),
}

for name, r in cases.items():
    adv = group_advantage(r)
    print(f'{name:>20}: rewards={r.tolist()}  -> advantages={[round(x, 2) for x in adv.tolist()]}')
# Think: which case maximises the policy-update magnitude?
# Which case is *invisible* to GRPO (zero gradient)?

## Progresión esperada — `CANONICAL_PROMPT`

Resultados **reales** del pipeline completo (OPT-350M, RTX 5070 Ti):

| Etapa | Salida sobre `"Explain quantum computing to a 10-year-old."` | Tiempo de entrenamiento |
|---|---|---|
| **Base** (01) | `What's quantum computing?` | — |
| **SFT** (02) | `quantum computing is a technology that uses a quantum field to create a quantum computer, …` | ~4 min |
| **DPO** (05) | `In quantum computing, each particle in the universe is a part of a quantum system. The particle is called a qubit. …` | 11 s (post-SFT) |
| **PPO** (04) | *(vacío — colapso del policy)* | 22 s (post-SFT + RM) |
| **GRPO** (este) | `Quantum computing is the process of computing information using a quantum computer. … Quantum computing can be used to perform many different types of calculations, including computing the motion of atoms, or determining the state of an electron.` | 30 s (post-SFT + RM) |

Observaciones sobre la salida de GRPO:

- **Vocabulario más rico que SFT**: aparece "atoms", "electrons", "state of an electron" — términos que el SFT no priorizaba y que la heurística del RM premia indirectamente (longitud, palabras técnicas).
- **Sigue habiendo repetición temática** ("similar to what is used in quantum computing, but the quantum computing process uses quantum information") — el modelo encuentra patrones que pasan el RM y se queda en ellos.
- **No colapsó como PPO** — el baseline del grupo es más estable que la cabeza de valor mal entrenada de PPO. Este es exactamente el argumento de DeepSeek para GRPO.

PPO y GRPO se compararon en igualdad de condiciones — mismos SFT + RM, mismo número de iteraciones (10). GRPO produjo respuestas coherentes; PPO colapsó en el prompt canónico. A *este* tamaño de modelo y régimen de muestras, **GRPO domina empíricamente** — lo que coincide con el patrón observado por DeepSeek-R1.

## Fin de la serie — y un cierre coherente

Has visto cuatro formas de alinear un modelo de lenguaje, todas evaluadas sobre el mismo `CANONICAL_PROMPT`:

1. **Base** (`01`) — el LLM no entiende que se le hizo una pregunta.
2. **SFT** (`02`) — aprende el *formato* de respuesta a partir de demostraciones humanas (Dolly-15k).
3. **PPO** (`04`) y **GRPO** (`06`) — afinan el contenido optimizando contra el RM entrenado en `03`.
4. **DPO** (`05`) — alternativa a PPO/GRPO; mismos pares de preferencias sin RM separado.

**Coherencia del repo:** los cuatro trainers toman el mismo modelo SFT (`checkpoints/sft/adapter`) como punto de partida y emiten su propio adaptador (`checkpoints/{ppo,dpo,grpo}/policy`). Si entrenas las cuatro etapas en orden, puedes abrir un notebook nuevo, cargar los cuatro adaptadores secuencialmente, y comparar las salidas sobre `EVAL_PROMPTS` lado a lado.

Para extensiones, mira la sección *What this codebase is not* del README principal.